<a href="https://colab.research.google.com/github/mayankrohilla-tech/Banking-Term-Deposit/blob/main/SimpleRAG_System.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Building an END-TO-END RAG Chain

Step1 - initialize the Chroma DB Connection /Pinecone / Weaviate / Qdrant / Milvus/ FISSA etc

Step2 - Create a Retriever Object

Step3 - Initialize a Chat Prompt Template

Step4 - Initialize a Generator (i.e. Chat Model)

Step5 - Initialize a Output Parser

Step6 - Define a RAG Chain

Step7 - Invoke the Chain

## Install certain dependencies - LangChain, LLM Model

In [1]:
#!pip install langchain==0.3.10

In [4]:
#!pip install langchain-openai==0.2.12

In [6]:
#!pip install langchain-community==0.3.11

In [8]:
# json file
# !pip install jq==1.8.0

In [10]:
# pdf file
# !pip install pymupdf==1.25.1

# Install Chroma Vector DB and LangChain wrapper

In [1]:
#!pip install langchain-chroma==0.1.4

In [ ]:
# Enter OPENAI API Key
#from getpass import getpass
#OPENAI_API_KEY = getpass("Enter your OPENAI API KEY here:")

In [11]:
# setup Environment variables

from google.colab import userdata
import os

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

## Step 1 - Initialize an Embedding Model to store into vector Database

In [13]:
from langchain_openai import OpenAIEmbeddings
embedding_model = OpenAIEmbeddings(model='text-embedding-3-small')

### Load and Process JSON Document and PDF Docs

In [14]:
from langchain.document_loaders import JSONLoader

loader = JSONLoader(file_path='/content/drive/MyDrive/Colab Notebooks/GenAI_Sundram Practice/rag_docs/wikidata_rag_demo.jsonl',
                    jq_schema='.',
                    text_content=False,
                    json_lines=True)
wiki_docs = loader.load()
print(len(wiki_docs))

1801


In [15]:
wiki_docs[1]

Document(metadata={'source': '/content/drive/MyDrive/Colab Notebooks/GenAI_Sundram Practice/rag_docs/wikidata_rag_demo.jsonl', 'seq_num': 2}, page_content='{"id": "86394", "title": "Dattatreya", "paragraphs": ["Dattatreya is the God who is an incarnation of the Divine Trinity Brahma, Vishnu and Siva. The word Datta means \\"Given\\", Datta is called so because the divine trinity have \\"given\\" themselves in the form of a son to the sage couple Guru Atri and Mata Anusuya. He is the son of Guru Atri, hence the name \\"Atreya.\\"", "In the Nath tradition, Dattatreya is seen as an Avatar or incarnation of the Lord Shiva and as the Adi-Guru (First Teacher) of the Adi-Nath sampradaya of the Nathas. Although Dattatreya was at first a \\"Lord of Yoga\\" with Tantric traits, he was adapted and assimilated into the more devotional cults; while still worshiped by millions of Hindus, he is approached more as a benevolent God than as a teacher of the highest essence of Indian thought.", "Though t

In [18]:
import json
from langchain.docstore.document import Document

wiki_docs_processed = []

for doc in wiki_docs:
  doc = json.loads(doc.page_content)
  metadata = {
      "title": doc["title"],
      "id": doc["id"],
      "source": "wikipedia"
  }
  data = ' '.join(doc["paragraphs"])
  wiki_docs_processed.append(Document(page_content=data, metadata=metadata))

In [19]:
wiki_docs_processed[1234]

Document(metadata={'title': 'Horatio Alger, Jr.', 'id': '358744', 'source': 'wikipedia'}, page_content='Horatio Alger, Jr. (January 13, 1832 – July 18, 1899) was an American writer. He wrote magazine stories and poems, a few novels for adults, and 100 plus boys\' books. His boys\' books were hugely popular. Alger was born in Massachusetts, and attended Harvard College. He became a Unitarian minister, but his career as a clergyman was brief. It ended when his congregation charged him with child molestation. Criminal charges were not placed against him, but his career in the church was finished. He moved to New York City to become a professional writer. In 1868, Alger found his place in the literary world with his fourth boys\' book, "Ragged Dick". This book is about a poor shoe shine boy in New York City who rises to middle class comfort and security through hard work, honesty, and a little luck. The book was a great success.')

### Load and process PDF Documents

In [21]:
from langchain.document_loaders import PyMuPDFLoader   # Document Loader
from langchain.text_splitter import RecursiveCharacterTextSplitter   # Chunking Strategy


In [22]:
def create_simple_chunks(file_path, chunk_size=500, chunk_overlap=40):
  print("Loading pages: ", file_path)
  loader = PyMuPDFLoader(file_path)
  doc_pages = loader.load()

  print("Chunking pages :", file_path)
  splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
  doc_chunks = splitter.split_documents(doc_pages)

  print("Finishing Loading and Processing steps with PDF Dataset :", file_path)
  print("******************************************")
  return doc_chunks

In [24]:
from glob import glob
pdf_files = glob('/content/drive/MyDrive/Colab Notebooks/GenAI_Sundram Practice/rag_docs/*.pdf')
pdf_files

['/content/drive/MyDrive/Colab Notebooks/GenAI_Sundram Practice/rag_docs/resnet_paper.pdf',
 '/content/drive/MyDrive/Colab Notebooks/GenAI_Sundram Practice/rag_docs/cnn_paper.pdf',
 '/content/drive/MyDrive/Colab Notebooks/GenAI_Sundram Practice/rag_docs/attention_paper.pdf',
 '/content/drive/MyDrive/Colab Notebooks/GenAI_Sundram Practice/rag_docs/vision_transformer.pdf']

In [25]:
paper_docs = []
for my_doc in pdf_files:
  paper_docs.extend(create_simple_chunks(file_path=my_doc, chunk_size=500, chunk_overlap=40))

Loading pages:  /content/drive/MyDrive/Colab Notebooks/GenAI_Sundram Practice/rag_docs/resnet_paper.pdf
Chunking pages : /content/drive/MyDrive/Colab Notebooks/GenAI_Sundram Practice/rag_docs/resnet_paper.pdf
Finishing Loading and Processing steps with PDF Dataset : /content/drive/MyDrive/Colab Notebooks/GenAI_Sundram Practice/rag_docs/resnet_paper.pdf
******************************************
Loading pages:  /content/drive/MyDrive/Colab Notebooks/GenAI_Sundram Practice/rag_docs/cnn_paper.pdf
Chunking pages : /content/drive/MyDrive/Colab Notebooks/GenAI_Sundram Practice/rag_docs/cnn_paper.pdf
Finishing Loading and Processing steps with PDF Dataset : /content/drive/MyDrive/Colab Notebooks/GenAI_Sundram Practice/rag_docs/cnn_paper.pdf
******************************************
Loading pages:  /content/drive/MyDrive/Colab Notebooks/GenAI_Sundram Practice/rag_docs/attention_paper.pdf
Chunking pages : /content/drive/MyDrive/Colab Notebooks/GenAI_Sundram Practice/rag_docs/attention_paper.pd

In [26]:
print(len(paper_docs))

444


In [27]:
paper_docs[0]

Document(metadata={'source': '/content/drive/MyDrive/Colab Notebooks/GenAI_Sundram Practice/rag_docs/resnet_paper.pdf', 'file_path': '/content/drive/MyDrive/Colab Notebooks/GenAI_Sundram Practice/rag_docs/resnet_paper.pdf', 'page': 0, 'total_pages': 12, 'format': 'PDF 1.5', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'creator': 'LaTeX with hyperref package', 'producer': 'pdfTeX-1.40.12', 'creationDate': 'D:20151211011345Z', 'modDate': 'D:20151211011345Z', 'trapped': ''}, page_content='Deep Residual Learning for Image Recognition\nKaiming He\nXiangyu Zhang\nShaoqing Ren\nJian Sun\nMicrosoft Research\n{kahe, v-xiangz, v-shren, jiansun}@microsoft.com\nAbstract\nDeeper neural networks are more difﬁcult to train. We\npresent a residual learning framework to ease the training\nof networks that are substantially deeper than those used\npreviously. We explicitly reformulate the layers as learn-\ning residual functions with reference to the layer inputs, in-')

In [29]:
print(len(wiki_docs))
print(len(paper_docs))

1801
444


In [31]:
total_docs = wiki_docs + paper_docs
print(len(total_docs))

2245


### Index Document chunks and Embeddings in vector DataBase

In [32]:
from langchain_chroma import Chroma

In [33]:
# Create vector DB of docs and embedding
## Please note below steps use / execute only one time

chroma_db = Chroma.from_documents(
    documents = total_docs,
    embedding = embedding_model,
    persist_directory='./my_db',
    collection_metadata = {"hnsw:space": "cosine"}
)
# HNSW - Hierarchical Navigable Small World

ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


### Load vector DB from your repository/disk

In [34]:
chroma_db

## This is the part of RAG

# Step 2 - Create a Retriever Object

we use simple cosine similarity here and retrieve the top 5 similar documents based on the user input query

In [35]:
similarity_retriever = chroma_db.as_retriever(search_type="similarity", search_kwargs={"k":5})

In [37]:
from IPython.display import display, Markdown

def display_docs(docs):
  for doc in docs:
    print("Metadata:", doc.metadata)
    print("Content Brief:")
    display(Markdown(doc.page_content[:4000]))
    print("******************************")

# Step 3  - Initialize a Chat Prompt Template

In [38]:
query1 = "what is self attention model?"
query2 = "what is CNN model?"
query3 = "what is all about resent paper?"

# Step4 - Initialize a Generator (i.e. Chat Model)

In [39]:
top_docs = similarity_retriever.invoke(query1)
display_docs(top_docs)

ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


Metadata: {'author': '', 'creationDate': 'D:20210604001958Z', 'creator': 'LaTeX with hyperref', 'file_path': '/content/drive/MyDrive/Colab Notebooks/GenAI_Sundram Practice/rag_docs/vision_transformer.pdf', 'format': 'PDF 1.5', 'keywords': '', 'modDate': 'D:20210604001958Z', 'page': 1, 'producer': 'pdfTeX-1.40.21', 'source': '/content/drive/MyDrive/Colab Notebooks/GenAI_Sundram Practice/rag_docs/vision_transformer.pdf', 'subject': '', 'title': '', 'total_pages': 22, 'trapped': ''}
Content Brief:


of self-attention, e.g. by augmenting feature maps for image classiﬁcation (Bello et al., 2019) or by
further processing the output of a CNN using self-attention, e.g. for object detection (Hu et al., 2018;
Carion et al., 2020), video processing (Wang et al., 2018; Sun et al., 2019), image classiﬁcation (Wu
et al., 2020), unsupervised object discovery (Locatello et al., 2020), or uniﬁed text-vision tasks (Chen
et al., 2020c; Lu et al., 2019; Li et al., 2019).

******************************
Metadata: {'author': '', 'creationDate': 'D:20230803000729Z', 'creator': 'LaTeX with hyperref', 'file_path': '/content/drive/MyDrive/Colab Notebooks/GenAI_Sundram Practice/rag_docs/attention_paper.pdf', 'format': 'PDF 1.5', 'keywords': '', 'modDate': 'D:20230803000729Z', 'page': 1, 'producer': 'pdfTeX-1.40.25', 'source': '/content/drive/MyDrive/Colab Notebooks/GenAI_Sundram Practice/rag_docs/attention_paper.pdf', 'subject': '', 'title': '', 'total_pages': 15, 'trapped': ''}
Content Brief:


reduced to a constant number of operations, albeit at the cost of reduced effective resolution due
to averaging attention-weighted positions, an effect we counteract with Multi-Head Attention as
described in section 3.2.
Self-attention, sometimes called intra-attention is an attention mechanism relating different positions
of a single sequence in order to compute a representation of the sequence. Self-attention has been

...

******************************
Metadata: {'author': '', 'creationDate': 'D:20230803000729Z', 'creator': 'LaTeX with hyperref', 'file_path': '/content/drive/MyDrive/Colab Notebooks/GenAI_Sundram Practice/rag_docs/attention_paper.pdf', 'format': 'PDF 1.5', 'keywords': '', 'modDate': 'D:20230803000729Z', 'page': 5, 'producer': 'pdfTeX-1.40.25', 'source': '/content/drive/MyDrive/Colab Notebooks/GenAI_Sundram Practice/rag_docs/attention_paper.pdf', 'subject': '', 'title': '', 'total_pages': 15, 'trapped': ''}
Content Brief:


because it may allow the model to extrapolate to sequence lengths longer than the ones encountered
during training.
4
Why Self-Attention
In this section we compare various aspects of self-attention layers to the recurrent and convolu-
tional layers commonly used for mapping one variable-length sequence of symbol representations
(x1, ..., xn) to another sequence of equal length (z1, ..., zn), with xi, zi ∈Rd, such as a hidden

******************************
Metadata: {'author': '', 'creationDate': 'D:20210604001958Z', 'creator': 'LaTeX with hyperref', 'file_path': '/content/drive/MyDrive/Colab Notebooks/GenAI_Sundram Practice/rag_docs/vision_transformer.pdf', 'format': 'PDF 1.5', 'keywords': '', 'modDate': 'D:20210604001958Z', 'page': 7, 'producer': 'pdfTeX-1.40.21', 'source': '/content/drive/MyDrive/Colab Notebooks/GenAI_Sundram Practice/rag_docs/vision_transformer.pdf', 'subject': '', 'title': '', 'total_pages': 22, 'trapped': ''}
Content Brief:


attention distance increases with network depth. Globally, we ﬁnd that the model attends to image
regions that are semantically relevant for classiﬁcation (Figure 6).
4.6
SELF-SUPERVISION
Transformers show impressive performance on NLP tasks. However, much of their success stems
not only from their excellent scalability but also from large scale self-supervised pre-training (Devlin
8

******************************


In [40]:
query = "What is Machine Learning?"
top_docs = similarity_retriever.invoke(query)
display_docs(top_docs)

Metadata: {'seq_num': 1712, 'source': '/content/drive/MyDrive/Colab Notebooks/GenAI_Sundram Practice/rag_docs/wikidata_rag_demo.jsonl'}
Content Brief:


{"id": "564928", "title": "Machine learning", "paragraphs": ["Machine learning gives computers the ability to learn without being explicitly programmed (Arthur Samuel, 1959). It is a subfield of computer science.", "The idea came from work in artificial intelligence. Machine learning explores the study and construction of algorithms which can learn and make predictions on data. Such algorithms follow programmed instructions, but can also make predictions or decisions based on data. They build a model from sample inputs.", "Machine learning is done where designing and programming explicit algorithms cannot be done. Examples include spam filtering, detection of network intruders or malicious insiders working towards a data breach, optical character recognition (OCR), search engines and computer vision."]}

******************************
Metadata: {'seq_num': 1278, 'source': '/content/drive/MyDrive/Colab Notebooks/GenAI_Sundram Practice/rag_docs/wikidata_rag_demo.jsonl'}
Content Brief:


{"id": "663523", "title": "Deep learning", "paragraphs": ["Deep learning (also called deep structured learning or hierarchical learning) is a kind of machine learning, which is mostly used with certain kinds of neural networks. As with other kinds of machine-learning, learning sessions can be unsupervised, semi-supervised, or supervised. In many cases, structures are organised so that there is at least one intermediate layer (or hidden layer), between the input layer and the output layer.", "Certain tasks, such as as recognizing and understanding speech, images or handwriting, is easy to do for humans. However, for a computer, these tasks are very difficult to do. In a multi-layer neural network (having more than two layers), the information processed will become more abstract with each added layer.", "Deep learning models are inspired by information processing and communication patterns in biological nervous systems; they are different from the structural and functional properties of biological brains (especially the human brain) in many ways, which make them incompatible with neuroscience evidences."]}

******************************
Metadata: {'seq_num': 1236, 'source': '/content/drive/MyDrive/Colab Notebooks/GenAI_Sundram Practice/rag_docs/wikidata_rag_demo.jsonl'}
Content Brief:


...

******************************
Metadata: {'seq_num': 840, 'source': '/content/drive/MyDrive/Colab Notebooks/GenAI_Sundram Practice/rag_docs/wikidata_rag_demo.jsonl'}
Content Brief:


{"id": "6360", "title": "Artificial intelligence", "paragraphs": ["Artificial intelligence (AI) is the ability of a computer program or a machine to think and learn. It is also a field of study which tries to make computers \"smart\". They work on their own without being encoded with commands. John McCarthy came up with the name \"Artificial Intelligence\" in 1955.", "In general use, the term \"artificial intelligence\" means a programme which mimics human cognition. At least some of the things we associate with other minds, such as learning and problem solving can be done by computers, though not in the same way as we do. Andreas Kaplan and Michael Haenlein define AI as a system\u2019s ability to correctly interpret external data, to learn from such data, and to use those learnings to achieve specific goals and tasks through flexible adaptation.", "An ideal (perfect) intelligent machine is a flexible agent which perceives its environment and takes actions to maximize its chance of success at some goal or objective. As machines become increasingly capable, mental faculties once thought to require intelligence are removed from the definition. For example, optical character recognition is no longer perceived as an example of \"artificial intelligence\": it is just a routine technology."]}

******************************
Metadata: {'author': '', 'creationDate': 'D:20151203014807Z', 'creator': 'LaTeX with hyperref package', 'file_path': '/content/drive/MyDrive/Colab Notebooks/GenAI_Sundram Practice/rag_docs/cnn_paper.pdf', 'format': 'PDF 1.5', 'keywords': '', 'modDate': 'D:20151203014807Z', 'page': 1, 'producer': 'pdfTeX-1.40.12', 'source': '/content/drive/MyDrive/Colab Notebooks/GenAI_Sundram Practice/rag_docs/cnn_paper.pdf', 'subject': '', 'title': '', 'total_pages': 11, 'trapped': ''}
Content Brief:


training.
Unsupervised learning differs in that the training set does not include any la-
bels. Success is usually determined by whether the network is able to reduce or
increase an associated cost function. However, it is important to note that most
image-focused pattern-recognition tasks usually depend on classiﬁcation using
supervised learning.
Convolutional Neural Networks (CNNs) are analogous to traditional ANNs

******************************


# Build the RAG Pipeline

# Step5 - Initialize a Output Parser

# Step6 - Define a RAG Chain

In [51]:
# Augmentation method - Enhanced Prompt

from langchain_core.prompts import ChatPromptTemplate


rag_prompt = """You are an assistant who is an expert in question-answering tasks.
                Your rule here is to answer the following question using only the following pieces of
                retrieval context. If the answer is not in the context, do not make up answers, just say that I don't know.
                Keep the answer detailed and well formatted based on the information from the context.
                You can also share the citation of the response as from where this response comes from.

                Question:
                {question}

                Context:
                {context}

                Answer:
                """
rag_prompt_template = ChatPromptTemplate.from_template(rag_prompt)

In [52]:
from langchain_core.runnables import RunnablePassthrough
from langchain_openai import ChatOpenAI

chatgpt = ChatOpenAI(model_name='gpt-4o-mini', temperature=0)

def format_docs(docs):
  return "\n\n".join(doc.page_content for doc in docs)

qa_rag_chain = (
    {"context":(similarity_retriever | format_docs),
    "question": RunnablePassthrough()}
    | rag_prompt_template
    | chatgpt)

# Step7 - Invoke the Chain

------------------

In [53]:
query = "What is Machine Learning?"
result = qa_rag_chain.invoke(query)
display(Markdown(result.content))

Machine learning is a subfield of computer science that provides computers the ability to learn without being explicitly programmed. The concept originated from work in artificial intelligence and involves the study and construction of algorithms that can learn from data and make predictions. These algorithms follow programmed instructions but can also make decisions based on the data they process, building a model from sample inputs.

Machine learning is particularly useful in scenarios where designing and programming explicit algorithms is not feasible. Examples of its applications include spam filtering, detection of network intruders or malicious insiders, optical character recognition (OCR), search engines, and computer vision.

(Source: "Machine learning")

In [54]:
query = "Who is PM of India?"
result = qa_rag_chain.invoke(query)
display(Markdown(result.content))

I don't know.